# Videos — Silver Transformation

- Bronze → Silver
- Inspect Bronze video data
- Transform to Silver schema
- Write Silver Delta table
- Validate Silver data

## 1. Inspect Bronze data

In [0]:
%sql
-- Preview Bronze data
SELECT *
FROM youtube_content_intelligence.bronze.brz_videos
LIMIT 5;

In [0]:
%sql
-- Inspect Bronze schema
DESCRIBE TABLE youtube_content_intelligence.bronze.brz_videos;

## 2. Transform Bronze data

In [0]:
from pyspark.sql.functions import col, regexp_extract, round, to_timestamp, when

# Load Bronze table
df_bronze = spark.table("youtube_content_intelligence.bronze.brz_videos")

# Convert duration to minutes
hours = regexp_extract("duration", r"(\d+)H", 1)
minutes = regexp_extract("duration", r"(\d+)M", 1)
seconds = regexp_extract("duration", r"(\d+)S", 1)

duration_minutes = round(
    (
        when(hours != "", hours.cast("double")).otherwise(0.0) * 60
        + when(minutes != "", minutes.cast("double")).otherwise(0.0)
        + when(seconds != "", seconds.cast("double")).otherwise(0.0) / 60
    ),
    2
)

df_silver = df_bronze.select(
    col("video_id"),
    col("channel_id"),
    col("category_id").cast("int").alias("category_id"),
    col("title"),
    to_timestamp("published_at").alias("published_at"),
    duration_minutes.alias("duration_minutes"),
    col("video_url"),
    to_timestamp("collected_at").alias("collected_at")
)

In [0]:
display(df_silver.limit(5))

In [0]:
df_silver.printSchema()

## 3. Write to Silver

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable(
    "youtube_content_intelligence.silver.slv_videos"
)

## 4. Validate Silver table

In [0]:
%sql
-- Preview Silver data
SELECT *
FROM youtube_content_intelligence.silver.slv_videos
LIMIT 5;

In [0]:
%sql
-- Check row count
SELECT COUNT(*) AS row_count
FROM youtube_content_intelligence.silver.slv_videos;

In [0]:
%sql
-- Check duplicate video IDs
SELECT video_id, COUNT(*) AS count
FROM youtube_content_intelligence.silver.slv_videos
GROUP BY video_id
HAVING COUNT(*) > 1;